# Genomic VEP — Training Notebook

**Setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have parquet files in Google Drive (genomic_vep folder)
3. Run all cells

## 1. Clone repo & install deps

In [ ]:
import os

# Clone repo (skip if already cloned)
if not os.path.exists("genomic-vep"):
    !git clone https://github.com/theomalaper/genomic-vep.git

%cd genomic-vep
!pip install -q "numpy>=1.24,<2.0" --force-reinstall
!pip install -q -r requirements.txt

## 2. Verify GPU & copy data

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Runtime → Change runtime type → T4 GPU")
    
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy parquet files from Drive
!mkdir -p data
!cp /content/drive/MyDrive/genomic_vep/*.parquet data/

for split in ["train", "val", "test"]:
    path = f"data/{split}.parquet"
    if os.path.exists(path):
        print(f"  {split}.parquet: {os.path.getsize(path) / 1e6:.1f} MB")
    else:
        print(f"  MISSING: {path}")

# Save checkpoints directly to Drive (survives session disconnects)
import os
os.environ["CHECKPOINT_DIR"] = "/content/drive/MyDrive/genomic_vep/checkpoints"
os.makedirs(os.environ["CHECKPOINT_DIR"], exist_ok=True)
print(f"Checkpoints will save to: {os.environ['CHECKPOINT_DIR']}")

## 4. Train

In [ ]:
from src.model.train import train

model = train(epochs=10, batch_size=32, lr=1e-3, pos_weight=2.0)

## 5. Test evaluation

In [ ]:
from src.model.train import evaluate
from src.data.dataset import create_dataloaders

_, _, test_loader = create_dataloaders(batch_size=32)
test_auroc = evaluate(model, test_loader)
print(f"Test AUROC: {test_auroc:.4f}")

## 6. Download checkpoint

In [ ]:
# Checkpoints are already on Drive — just print the path
import os
ckpt_dir = os.environ.get("CHECKPOINT_DIR", "checkpoints")
for f in os.listdir(ckpt_dir):
    size = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

In [ ]:
from google.colab import files
files.download("checkpoints/best_model.pth")